In [1]:
from pathlib import Path
import shutil
import hashlib
from collections import Counter

import pandas as pd
from sklearn.model_selection import train_test_split

# =========================
# CONFIG
# =========================
input_root = Path(r"D:\Project\DAP_paper\datasets\processed\GeoDE")
output_root = Path(r"D:\Project\DAP_paper\datasets\processed\GeoDE_split")

seed = 42

# Paper-style GeoDE split
train_per_region = 4970

# Keep original GeoDE safe
copy_files = True   # True = copy, False = move

valid_exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# =========================
# COLLECT IMAGE PATHS
# =========================
rows = []

for region_dir in sorted(input_root.iterdir()):
    if not region_dir.is_dir():
        continue

    region = region_dir.name

    for class_dir in sorted(region_dir.iterdir()):
        if not class_dir.is_dir():
            continue

        class_name = class_dir.name

        for img_path in class_dir.iterdir():
            if img_path.is_file() and img_path.suffix.lower() in valid_exts:
                rows.append({
                    "path": img_path,
                    "region": region,
                    "class": class_name
                })

df = pd.DataFrame(rows)

print("Total images:", len(df))
print("Regions:", sorted(df["region"].unique()))
print("Classes:", df["class"].nunique())

display(
    df.groupby(["region", "class"])
      .size()
      .reset_index(name="num_images")
)

Total images: 61925
Regions: ['Africa', 'Americas', 'EastAsia', 'Europe', 'SouthEastAsia', 'WestAsia']
Classes: 40


,region,class,num_images
0,Africa,backyard,670
1,Africa,bag,397
2,Africa,bicycle,257
3,Africa,boat,227
4,Africa,bus,240
...,...,...,...
235,WestAsia,toy,224
236,WestAsia,tree,226
237,WestAsia,truck,205
238,WestAsia,waste_container,231


In [2]:
def can_stratify(labels, train_size, test_size):
    counts = Counter(labels)
    num_classes = len(counts)

    if min(counts.values()) < 2:
        return False

    if train_size < num_classes:
        return False

    if test_size < num_classes:
        return False

    return True


def safe_copy_or_move(src, dst_dir, copy_files=True):
    dst_dir.mkdir(parents=True, exist_ok=True)

    dst = dst_dir / src.name

    # Avoid overwrite if duplicated filename exists
    if dst.exists():
        h = hashlib.md5(str(src).encode("utf-8")).hexdigest()[:8]
        dst = dst_dir / f"{src.stem}_{h}{src.suffix}"

    if copy_files:
        shutil.copy2(src, dst)
    else:
        shutil.move(str(src), str(dst))


split_rows = []

for region in sorted(df["region"].unique()):
    region_df = df[df["region"] == region].reset_index(drop=True)

    n_region = len(region_df)

    if n_region <= train_per_region:
        raise ValueError(
            f"Region {region} only has {n_region} images, "
            f"but train_per_region={train_per_region}"
        )

    y = region_df["class"]

    train_size = train_per_region
    temp_size = n_region - train_size

    stratify_y = y if can_stratify(y, train_size, temp_size) else None

    train_df, temp_df = train_test_split(
        region_df,
        train_size=train_size,
        random_state=seed,
        shuffle=True,
        stratify=stratify_y
    )

    # Split remaining images into val/test equally
    temp_y = temp_df["class"]

    val_size = len(temp_df) // 2
    test_size = len(temp_df) - val_size

    stratify_temp_y = temp_y if can_stratify(temp_y, val_size, test_size) else None

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.5,
        random_state=seed,
        shuffle=True,
        stratify=stratify_temp_y
    )

    for split_name, split_df in [
        ("train", train_df),
        ("val", val_df),
        ("test", test_df)
    ]:
        for _, row in split_df.iterrows():
            src = row["path"]
            dst_dir = output_root / split_name / row["region"] / row["class"]

            safe_copy_or_move(src, dst_dir, copy_files=copy_files)

            split_rows.append({
                "split": split_name,
                "region": row["region"],
                "class": row["class"],
                "filename": src.name
            })

    print(
        f"{region}: "
        f"train={len(train_df)}, val={len(val_df)}, test={len(test_df)}, total={n_region}"
    )

split_df = pd.DataFrame(split_rows)

print("=" * 60)
print("Done splitting GeoDE.")
display(split_df.head())

Africa: train=4970, val=3575, test=3576, total=12121
Americas: train=4970, val=2336, test=2337, total=9643
EastAsia: train=4970, val=2388, test=2389, total=9747
Europe: train=4970, val=3070, test=3071, total=11111
SouthEastAsia: train=4970, val=2790, test=2791, total=10551
WestAsia: train=4970, val=1891, test=1891, total=8752
Done splitting GeoDE.


,split,region,class,filename
0,train,Africa,backyard,South_Africa_backyard_53185.jpg
1,train,Africa,backyard,Nigeria_backyard_53644.jpg
2,train,Africa,spices,Egypt_spices_32910.jpg
3,train,Africa,house,Nigeria_house_41008.jpg
4,train,Africa,chair,Nigeria_chair_5398.jpg


In [4]:
# Overall split counts
summary = (
    split_df.groupby("split")
    .size()
    .reset_index(name="num_images")
)

display(summary)

# Region counts per split
region_summary = (
    split_df.groupby(["split", "region"])
    .size()
    .reset_index(name="num_images")
)

display(region_summary)

# Region x class counts per split
class_summary = (
    split_df.groupby(["split", "region", "class"])
    .size()
    .reset_index(name="num_images")
)

display(class_summary)

,split,num_images
0,test,16055
1,train,29820
2,val,16050


,split,region,num_images
0,test,Africa,3576
1,test,Americas,2337
2,test,EastAsia,2389
3,test,Europe,3071
4,test,SouthEastAsia,2791
5,test,WestAsia,1891
6,train,Africa,4970
7,train,Americas,4970
8,train,EastAsia,4970
9,train,Europe,4970


,split,region,class,num_images
0,test,Africa,backyard,198
1,test,Africa,bag,117
2,test,Africa,bicycle,76
3,test,Africa,boat,67
4,test,Africa,bus,71
...,...,...,...,...
715,val,WestAsia,toy,49
716,val,WestAsia,tree,49
717,val,WestAsia,truck,45
718,val,WestAsia,waste_container,50


In [5]:
count_table = class_summary.pivot_table(
    index=["split", "region"],
    columns="class",
    values="num_images",
    fill_value=0,
    aggfunc="sum"
).astype(int)

count_table["TOTAL"] = count_table.sum(axis=1)

display(count_table)

class                backyard  bag  bicycle  boat  bus  candle  car  chair  \
split region                                                                 
test  Africa              198  117       76    67   71      72   98    108   
      Americas             52   72       55    20   52      45   66     84   
      EastAsia             47   91       73    42   55      54   68     80   
      Europe               62  121       67    61   62      75  101     97   
      SouthEastAsia        57  157       62    62   56      63   62    136   
      WestAsia             46   57       51    35   44      50   53     60   
train Africa              275  163      105    93   98     100  136    150   
      Americas            112  154      118    43  112      97  141    177   
      EastAsia             98  189      152    89  114     112  141    166   
      Europe              101  195      108    99  101     121  162    156   
      SouthEastAsia       103  279      111   112  101     113  111    241   
      WestAsia            123  152      135    92  115     132  137    159   
val   Africa              197  117       76    67   71      72   97    107   
      Americas             53   72       55    21   53      46   66     83   
      EastAsia             47   90       73    43   54      54   67     80   
      Europe               63  121       66    62   63      74  100     96   
      SouthEastAsia        58  157       62    63   57      63   62    135   
      WestAsia             47   58       51    35   44      50   52     60   

class                cleaning_equipment  cooking_pot  ...  stove  \
split region                                          ...          
test  Africa                         84           79  ...    163   
      Americas                       66           51  ...     50   
      EastAsia                       75           56  ...     47   
      Europe                        100           84  ...     78   
      SouthEastAsia                  81           53  ...     70   
      WestAsia                       56           46  ...     43   
train Africa                        116          111  ...    227   
      Americas                      139          110  ...    106   
      EastAsia                      157          116  ...     97   
      Europe                        161          136  ...    126   
      SouthEastAsia                 144           95  ...    123   
      WestAsia                      147          123  ...    113   
val   Africa                         84           80  ...    163   
      Americas                       65           52  ...     50   
      EastAsia                       75           56  ...     47   
      Europe                        100           84  ...     78   
      SouthEastAsia                  80           54  ...     69   
      WestAsia                       56           47  ...     43   

class                streetlight_lantern  toothbrush  toothpaste_toothpowder  \
split region                                                                   
test  Africa                         102          76                      85   
      Americas                        50          82                      60   
      EastAsia                        51          81                      66   
      Europe                          62          75                      87   
      SouthEastAsia                   52          96                      61   
      WestAsia                        44          57                      45   
train Africa                         142         106                     118   
      Americas                       107         174                     126   
      EastAsia                       108         168                     137   
      Europe                         102         121                     141   
      SouthEastAsia                   92         170                     108   
      WestAsia                       1